# 04 · Evaluación Temporal

Objetivo (PDF sección 3.3): analizar el comportamiento y la posible degradación del rendimiento del modelo a lo largo del tiempo, evaluando sobre test.parquet, el bloque de datos que ningún notebook anterior había usado.

In [1]:
import sys
sys.path.append('../src')

import pandas as pd
import matplotlib.pyplot as plt

from models import get_models
from evaluation import compute_metrics, business_cost, evaluate_by_time_block

train_df = pd.read_parquet('../data/processed/train.parquet')
test_df = pd.read_parquet('../data/processed/test.parquet')

drop_cols = ['TransactionID', 'isFraud', 'TransactionDT', 'day']
feature_cols = [c for c in train_df.columns if c not in drop_cols]

BEST_MODEL_NAME = 'xgboost'      # elegido en el notebook 03 (mejor ROC-AUC en val)
BEST_THRESHOLD = 0.07            # umbral costo-optimo hallado en el notebook 03

model = get_models()[BEST_MODEL_NAME]
model.fit(train_df[feature_cols], train_df['isFraud'])

print('train:', train_df.shape, ' test:', test_df.shape)
print('modelo:', BEST_MODEL_NAME, ' umbral:', BEST_THRESHOLD)

train: (354324, 401)  test: (118108, 401)
modelo: xgboost  umbral: 0.07


Se reentrena el mismo modelo y la misma configuración elegidos en el notebook 03 (mismo train.parquet, sin volver a tocar val.parquet ni test.parquet para ajustar nada) y se reutiliza el umbral costo-óptimo que ya se había encontrado. Este notebook solo evalúa, no vuelve a elegir modelo ni umbral con datos de test, porque eso sería fuga de información hacia la evaluación final.

## 1. Evaluación global en test (una sola vez)

In [1]:
X_test, y_test = test_df[feature_cols], test_df['isFraud']
proba_test = model.predict_proba(X_test)[:, 1]
pred_test = (proba_test >= BEST_THRESHOLD).astype(int)

metrics_test = compute_metrics(y_test, pred_test, proba_test)
metrics_test['cost'] = business_cost(y_test, pred_test)
print('Metricas en test (nunca antes visto):')
print(metrics_test)

metrics_val_recordatorio = {'f1': 0.4952, 'roc_auc': 0.9206}
print()
print('(referencia: en val estas mismas metricas fueron f1=0.4952, roc_auc=0.9206)')

Metricas en test (nunca antes visto):
{'f1': 0.40555927662424646, 'precision': 0.30736040609137055, 'recall': 0.5959645669291339, 'roc_auc': 0.8863326286967015, 'cost': 278910.0}

(referencia: en val estas mismas metricas fueron f1=0.4952, roc_auc=0.9206)


## 2. Evaluación por bloque temporal

Se divide `test` (días 140-181) en 8 bloques cronológicos iguales y se calcula F1, ROC-AUC y costo de negocio en cada uno, para ver si el rendimiento se mantiene o se degrada con el tiempo.

In [1]:
test_eval_df = test_df.copy()
test_eval_df['y_true'] = y_test
test_eval_df['y_pred'] = pred_test
test_eval_df['y_proba'] = proba_test

block_metrics = evaluate_by_time_block(
    test_eval_df, y_true_col='y_true', y_pred_col='y_pred',
    y_proba_col='y_proba', dt_col='TransactionDT', n_blocks=8,
)

# costo de negocio por bloque (no esta en evaluate_by_time_block, se agrega aqui)
costs = []
for block, group in test_eval_df.groupby(pd.qcut(test_eval_df['TransactionDT'], q=8, labels=False, duplicates='drop')):
    costs.append(business_cost(group['y_true'], group['y_pred']))
block_metrics['cost'] = costs

print(block_metrics[['block', 'day_min', 'day_max', 'n', 'f1', 'roc_auc', 'cost']])

   block  day_min  day_max      n        f1   roc_auc     cost
0      0      140      145  14764  0.401150  0.878135  39480.0
1      1      145      151  14763  0.349501  0.885233  34680.0
2      2      151      155  14764  0.406227  0.899497  30390.0
3      3      155      160  14763  0.352518  0.881132  31980.0
4      4      160      165  14764  0.360158  0.868918  36285.0
5      5      165      170  14763  0.440076  0.880216  36030.0
6      6      170      175  14763  0.460591  0.903309  33405.0
7      7      175      181  14764  0.454277  0.892580  36660.0


In [1]:
fig, axes = plt.subplots(3, 1, figsize=(10, 8), sharex=True)
axes[0].plot(block_metrics['block'], block_metrics['f1'], marker='o', color='#0284c7')
axes[0].set_ylabel('F1')
axes[0].set_title('Desempeño por bloque temporal en test (xgboost, umbral 0.07)')
axes[1].plot(block_metrics['block'], block_metrics['roc_auc'], marker='o', color='#16a34a')
axes[1].set_ylabel('ROC-AUC')
axes[2].plot(block_metrics['block'], block_metrics['cost'], marker='o', color='#b91c1c')
axes[2].set_ylabel('Costo de negocio ($)')
axes[2].set_xlabel('Bloque temporal (0=mas antiguo, 7=mas reciente)')
plt.tight_layout()
plt.savefig('../figures/07_temporal_degradation.png', dpi=110)
plt.show()

El desempeño en test (F1 de 0.406, ROC-AUC de 0.886, costo de $278,910) es peor que en val (F1 de 0.495, ROC-AUC de 0.921, costo de $254,190), una caída relativa de F1 del 18%. Esto supera el umbral del 5% de degradación que definimos en el checkpoint 1 para disparar el reentrenamiento. Si este sistema estuviera en producción, la política de monitoreo ya habría activado una alerta al pasar de val a test. Es evidencia real, no solo teórica, de la no estacionariedad que ya habíamos encontrado en el EDA del notebook 01: tasa de fraude cambiante y un mix de ProductCD distinto entre periodos.

![Desempeño por bloque temporal](../figures/07_temporal_degradation.png)

Dentro de los cerca de 41 días de test, F1 y ROC-AUC oscilan sin una tendencia monótona de degradación. Bajan en los bloques 1, 3 y 4, pero se recuperan en los bloques 5 a 7 (el F1 termina en 0.454, más alto que al inicio). El costo de negocio por bloque se mueve entre $30,390 y $39,480 sin un patrón de deterioro sostenido. En otras palabras, el modelo no colapsa progresivamente dentro de esta ventana. La caída de rendimiento real está en el salto de val a test, que son dos periodos distintos, y no en un declive continuo dentro de test. Esto es consistente con el patrón de fraude ruidoso pero acotado que ya habíamos visto en el EDA, donde la tasa de fraude semanal subía y bajaba en vez de crecer de forma monótona.

## Conclusiones para el notebook 05

- Hay evidencia real, no solo hipotética, de degradación al cambiar de periodo. El propio umbral de alerta del 5% definido en el checkpoint 1 se habría activado.
- Dentro de una ventana de unos 40 días el rendimiento oscila pero no colapsa progresivamente. Esto sugiere que una política de reentrenamiento periódico (los escenarios A y B de la sección 7 del checkpoint 1) es razonable, sin necesitar reentrenar con una frecuencia extrema.
- El notebook 05 debe simular los dos escenarios de olvido (ventana deslizante fija y time-decay ponderado) sobre este mismo test.parquet, comparando si reentrenar con datos más recientes recupera el F1 o el costo que se perdió entre val y test.